In [30]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal
from pydantic import BaseModel,Field
from langchain_ollama import ChatOllama



In [31]:
class LLMschema(BaseModel):
    response:Literal["positive","negative"]=Field(description="sentiment analysis of the review")

    

In [32]:
review="""The movie had stunning visuals and an impressive soundtrack, but the story was painfully predictable. The lead actor delivered an outstanding performance, while the supporting characters felt underdeveloped. The first half was engaging, but the second half dragged on with unnecessary scenes. Despite its flaws, the emotional climax was satisfying and left a lasting impression. Overall, it was an enjoyable experience, though not as exceptional as the trailers suggested."""
prompt=f"perform sentiment analysis , and give me the feedback for the {review} as positive or negative. ONly as positive or negative "


In [33]:
class BasicStates(TypedDict):
    response:Literal["positive","negative"]
    description:str
    

In [34]:
model=ChatOllama(
    model='llama3.2',
    temperature=0
)
structured_model=model.with_structured_output(LLMschema)





In [35]:
def getsenti(state:BasicStates):
    result=structured_model.invoke(prompt)
    state['response']=result.response
    return {'response':result.response}




In [36]:
def no_description(state:BasicStates):
    prompt_description=f"give me description of two things of why do u think or on which basis u think the {review} is negative "
    response2=model.invoke(prompt_description)
    return {'description':response2.content}
    

In [37]:
def checkingfunction(state:BasicStates):
    if(state['response'] == "negative"):
        return "no"
    else:
        return "yes"

In [39]:
graph =StateGraph(BasicStates)
graph.add_node('getsentiment',getsenti)
graph.add_edge(START,'getsentiment')
graph.add_node("no_node",no_description)
graph.add_edge("no_node",END)
graph.add_conditional_edges('getsentiment',checkingfunction,{
    "yes":END,
    "no":"no_node"
})
workflow=graph.compile()
result=workflow.invoke({
    "review":review
})
print(result)


{'response': 'negative', 'description': "Based on the review, here are two possible reasons why the movie had stunning visuals and an impressive soundtrack, but the story was painfully predictable:\n\n1. **Overemphasis on technical aspects**: The reviewer highlights the movie's impressive visuals and soundtrack, suggesting that the filmmakers may have focused too much on showcasing their technical skills rather than developing a compelling narrative. This could be due to the filmmakers' desire to create a visually stunning film that would appeal to a broad audience, rather than taking risks on a more complex or original story.\n\n2. **Lack of attention to character development**: The reviewer notes that the supporting characters felt underdeveloped, which may indicate that the filmmakers prioritized plot over character development. This could be due to a lack of time or resources to flesh out the supporting characters, or a focus on creating a more straightforward narrative that doesn'